# 🛡️ RingShield: Google Colab to Confluent Cloud Kafka Streaming Demo

Welcome to the **RingShield** streaming demo! This interactive notebook demonstrates how an external client (running in Google Colab) can generate synthetic UPI transactions and stream them in real time to **Confluent Cloud Kafka**.

The local **RingShield Fraud Detection Consumer** reads these transactions from the `upi-transactions` topic, runs them through the explainable rule-based fraud detection engine, and publishes alerts to `fraud-alerts`.

---

### 🏗️ Streaming Architecture Overview

```text
 ┌────────────────────────────────────────────────────────┐
 │               Google Colab (This Notebook)              │
 │  • Generates Synthetic UPI Transactions                │
 │  • Enforces Schema Validation (contracts/schemas.py)   │
 │  • Publishes to Kafka with Key = sender_id             │
 └───────────────────────────┬────────────────────────────┘
                             │  SASL_SSL / JSON Payloads
                             ▼
 ┌────────────────────────────────────────────────────────┐
 │             Confluent Cloud Kafka Cluster              │
 │  • Input Topic:  upi-transactions (Partitioned by sender)│
 │  • Alert Topic:  fraud-alerts                          │
 └─────────────┬────────────────────────────▲─────────────┘
               │                            │
  Consumes tx  │                            │ Publishes confirmed
  stream       ▼                            │ fraud alerts
 ┌──────────────────────────────────────────┴─────────────┐
 │            Local Machine (Terminal Consumer)            │
 │  • python -m streaming.consumer                        │
 │  • Evaluates Rules:                                    │
 │      1. HIGH_VALUE_TRANSACTION (amount >= ₹100,000)    │
 │      2. RAPID_REPEATED_TRANSFERS (>= 5 in 60s)         │
 └────────────────────────────────────────────────────────┘
```

---

### 🔑 Key Design Concepts
1. **Per-Sender Partition Ordering**: Every transaction uses `sender_id` as the Kafka message key. Kafka hashes the key to assign the partition, guaranteeing that all transactions from the same sender arrive in sequential order on the same partition. This enables sliding-window fraud detection (like rapid bursts).
2. **At-Least-Once Delivery**: Downstream alerts are published and acknowledged by Kafka before message offsets are committed.
3. **Zero Secrets in Code**: Credentials are retrieved securely via Google Colab Secrets (`userdata`) or masked prompts (`getpass`). They are never printed or saved in outputs.

In [ ]:
# Step 0: Install Dependencies & Import Libraries
import os
import sys
import json
import time
from datetime import datetime, timezone
import getpass

# Install confluent-kafka if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Detected Google Colab environment. Installing confluent-kafka...")
    !pip install -q confluent-kafka
except ImportError:
    IN_COLAB = False
    print("Running in standard Python environment.")

from confluent_kafka import Producer, Consumer, TopicPartition, KafkaError
print("✅ Libraries imported successfully.")

## 1. Secure Confluent Cloud Credentials

You can supply your Confluent Cloud credentials in one of two secure ways:
- **Option A (Recommended for Google Colab)**: Use Colab Secrets (the 🔑 key icon on the left sidebar). Add secrets named `KAFKA_BOOTSTRAP_SERVERS`, `KAFKA_API_KEY`, and `KAFKA_API_SECRET`.
- **Option B (Interactive Prompt)**: Leave them blank in Secrets; the cell below will prompt you using masked input (`getpass.getpass`).

> ⚠️ **Security Policy**: Credentials will **never** be printed, logged, or saved in notebook output cells.

In [ ]:
# Step 1: Secure Credential Loading
def get_credential(name: str, prompt_text: str, is_secret: bool = False) -> str:
    # 1. Try Google Colab Secrets (userdata)
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val.strip()
    except Exception:
        pass

    # 2. Try OS environment variable
    val = os.getenv(name)
    if val:
        return val.strip()

    # 3. Prompt user interactively
    if is_secret:
        val = getpass.getpass(prompt_text)
    else:
        val = input(prompt_text)
    return val.strip()

BOOTSTRAP_SERVERS = get_credential(
    "KAFKA_BOOTSTRAP_SERVERS",
    "Enter Kafka Bootstrap Server (e.g. pkc-xxxxx.region.cloud.confluent.cloud:9092): "
)
API_KEY = get_credential(
    "KAFKA_API_KEY",
    "Enter Confluent Cloud API Key: "
)
API_SECRET = get_credential(
    "KAFKA_API_SECRET",
    "Enter Confluent Cloud API Secret: ",
    is_secret=True
)

INPUT_TOPIC = os.getenv("KAFKA_INPUT_TOPIC", "upi-transactions").strip()
ALERT_TOPIC = os.getenv("KAFKA_ALERT_TOPIC", "fraud-alerts").strip()

# Validate that required values are present
if not BOOTSTRAP_SERVERS or not API_KEY or not API_SECRET:
    raise ValueError("❌ Required Kafka credentials are missing. Please provide them to proceed.")

# Display a safe, masked confirmation (never reveals the secret)
masked_key = f"{API_KEY[:2]}****{API_KEY[-2:]}" if len(API_KEY) >= 4 else "******"
print("✅ Confluent Cloud configuration loaded:")
print(f"   • Bootstrap Server: {BOOTSTRAP_SERVERS.split(':')[0]}")
print(f"   • API Key:          {masked_key}")
print(f"   • Input Topic:      {INPUT_TOPIC}")
print(f"   • Alert Topic:      {ALERT_TOPIC}")

## 2. Event Contract & Schema Validation

To ensure complete compatibility with RingShield's backend contract (`contracts/schemas.py`), each transaction must conform to the following schema:

| Field | Type | Description |
| :--- | :---: | :--- |
| `transaction_id` | `str` | Unique transaction identifier (e.g. `colab_tx_1729680000_1001`) |
| `sender_id` | `str` | Sender VPA handle (e.g. `alice@okhdfcbank`) |
| `receiver_id` | `str` | Receiver VPA handle (must be different from `sender_id`) |
| `amount` | `float` / `int` | Transfer amount in INR (must be strictly $> 0$) |
| `timestamp` | `str` | ISO 8601 UTC timestamp string (e.g. `2026-10-23T12:00:00Z`) |

The validator below enforces these constraints before any message is queued.

In [ ]:
# Step 2: Schema Validation and Synthetic Transaction Generator
import time
from datetime import datetime, timezone

REQUIRED_FIELDS = {
    "transaction_id",
    "sender_id",
    "receiver_id",
    "amount",
    "timestamp",
}

def validate_transaction(transaction: dict) -> None:
    """Validate transaction against RingShield schema contract."""
    if not isinstance(transaction, dict):
        raise ValueError("Transaction must be a dictionary")

    missing = REQUIRED_FIELDS - transaction.keys()
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")

    if not isinstance(transaction["transaction_id"], str):
        raise ValueError("transaction_id must be a string")
    if not isinstance(transaction["sender_id"], str):
        raise ValueError("sender_id must be a string")
    if not isinstance(transaction["receiver_id"], str):
        raise ValueError("receiver_id must be a string")
    if transaction["sender_id"] == transaction["receiver_id"]:
        raise ValueError("sender_id and receiver_id must be different")

    if isinstance(transaction["amount"], bool) or not isinstance(transaction["amount"], (int, float)):
        raise ValueError("amount must be a number")
    if transaction["amount"] <= 0:
        raise ValueError("amount must be strictly greater than zero")

    if not isinstance(transaction["timestamp"], str):
        raise ValueError("timestamp must be a string")

def iso_utc_now() -> str:
    """Return current ISO 8601 timestamp in UTC."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def generate_synthetic_transactions() -> list[dict]:
    """Generate a deterministic suite of synthetic UPI transactions for testing:
    1. Normal everyday transactions (P2P and Merchant).
    2. High-value anomaly transaction (amount >= 100,000 INR).
    3. Rapid transfer burst (5 transactions within 30s from the same sender).
    """
    run_epoch = int(time.time())
    txs = []

    # 1. Normal Transactions (Expected: No fraud alerts)
    txs.append({
        "transaction_id": f"colab_norm_01_{run_epoch}",
        "sender_id": f"aarav_test_{run_epoch}@okhdfcbank",
        "receiver_id": "quickmart_groceries@icici",
        "amount": 450.0,
        "timestamp": iso_utc_now(),
    })
    txs.append({
        "transaction_id": f"colab_norm_02_{run_epoch}",
        "sender_id": f"priya_test_{run_epoch}@okaxis",
        "receiver_id": f"rohit_friend_{run_epoch}@oksbi",
        "amount": 1850.50,
        "timestamp": iso_utc_now(),
    })

    # 2. High-Value Fraud Anomaly (Expected: HIGH_VALUE_TRANSACTION alert)
    txs.append({
        "transaction_id": f"colab_high_val_{run_epoch}",
        "sender_id": f"vip_investor_{run_epoch}@paytm",
        "receiver_id": "suspicious_escrow@okaxis",
        "amount": 350000.0,  # Exceeds 100,000 threshold
        "timestamp": iso_utc_now(),
    })

    # 3. Rapid Repeated Transfers (5 transactions in < 30s from the same sender)
    burst_sender = f"burst_user_{run_epoch}@oksbi"
    for i in range(1, 6):
        txs.append({
            "transaction_id": f"colab_burst_{i}_{run_epoch}",
            "sender_id": burst_sender,
            "receiver_id": f"mule_collector_{i}@ybl",
            "amount": 1000.0 * i,
            "timestamp": iso_utc_now(),
        })

    # Validate all transactions prior to return
    for tx in txs:
        validate_transaction(tx)

    return txs

print("✅ Validator and synthetic transaction generator compiled.")

## 3. Dry-Run Simulation (Safe Local Test)

Before sending events over the internet to Confluent Cloud, run this dry-run cell. It validates and inspects the test payload locally without initiating any network connection.

In [ ]:
# Step 3: Run Dry-Run Simulation
sample_transactions = generate_synthetic_transactions()

print(f"Generated {len(sample_transactions)} synthetic transactions for demonstration:\n")
print(f"{'#':<3} | {'Transaction ID':<28} | {'Sender ID':<26} | {'Receiver ID':<26} | {'Amount (INR)':>12}")
print("-" * 105)

for idx, tx in enumerate(sample_transactions, start=1):
    print(f"{idx:<3} | {tx['transaction_id']:<28} | {tx['sender_id']:<26} | {tx['receiver_id']:<26} | ₹{tx['amount']:>11,.2f}")

print("-" * 105)
print("✅ Dry-run validation passed! All 8 transactions adhere to the RingShield schema.")
print("   • Expected Alerts in Consumer:")
print("     1. colab_high_val: [HIGH_VALUE_TRANSACTION] (Amount >= ₹100,000)")
print("     2. colab_burst_5:  [RAPID_REPEATED_TRANSFERS] (5th transfer from same sender in <60s)")

## 4. Start the Local RingShield Consumer

Before executing the live publish cell below, start the streaming consumer in a terminal on your local machine where RingShield is installed:

```bash
# In your local ringshield repository:
PYTHONUNBUFFERED=1 PYTHONPATH=. python3 -m streaming.consumer
```

You will see:
```text
Listening on topic: upi-transactions
Consumer group assigned partitions: [0, 1, 2, 3, 4, 5]
```

Keep that terminal open and visible while executing the next cell.

## 5. Live Kafka Publishing to Confluent Cloud

Set `PUBLISH_LIVE = True` below to stream the synthetic transactions to `upi-transactions`.

### 🛡️ Guardrails:
- `PUBLISH_LIVE` defaults to `False` to prevent accidental duplicate publishing when re-running notebook cells.
- Uses delivery callbacks to confirm that the Confluent Cloud broker has accepted and persisted each message.
- Sets `key = sender_id` to guarantee per-sender partition affinity.

In [ ]:
# Step 5: Live Publication to Confluent Cloud
# -------------------------------------------------------------
# SAFETY TOGGLE: Set to True when ready to publish live events.
PUBLISH_LIVE = False
# -------------------------------------------------------------

if not PUBLISH_LIVE:
    print("⏸️ PUBLISH_LIVE is False. Set PUBLISH_LIVE = True and rerun this cell to stream live to Kafka.")
else:
    # Build librdkafka client configuration
    conf = {
        "bootstrap.servers": BOOTSTRAP_SERVERS,
        "security.protocol": "SASL_SSL",
        "sasl.mechanisms": "PLAIN",
        "sasl.username": API_KEY,
        "sasl.password": API_SECRET,
        "acks": "all",
        "retries": 3,
        "retry.backoff.ms": 200,
        "delivery.timeout.ms": 15000,
        "request.timeout.ms": 10000,
    }

    producer = Producer(conf)
    delivery_stats = {"attempted": 0, "delivered": 0, "failed": 0}

    def on_delivery(err, msg):
        if err is not None:
            delivery_stats["failed"] += 1
            print(f"❌ [DELIVERY FAILED] {err.str()} (Code: {err.code()})")
        else:
            delivery_stats["delivered"] += 1
            print(f"✅ [DELIVERED] Topic: {msg.topic()} | Partition: {msg.partition()} | Offset: {msg.offset()}")

    transactions_to_send = generate_synthetic_transactions()
    print(f"🚀 Streaming {len(transactions_to_send)} transactions to '{INPUT_TOPIC}'...\n")

    for tx in transactions_to_send:
        delivery_stats["attempted"] += 1
        key = tx["sender_id"].encode("utf-8")
        payload = json.dumps(tx).encode("utf-8")

        producer.produce(
            topic=INPUT_TOPIC,
            key=key,
            value=payload,
            on_delivery=on_delivery,
        )
        producer.poll(0)
        time.sleep(0.3)  # Small pacing delay between events

    # Flush all remaining messages with timeout
    remaining = producer.flush(timeout=15.0)

    print("\n" + "=" * 55)
    print("         LIVE PUBLISHING SUMMARY")
    print("=" * 55)
    print(f"Total Attempted: {delivery_stats['attempted']}")
    print(f"Total Delivered: {delivery_stats['delivered']}")
    print(f"Total Failed:    {delivery_stats['failed']}")
    print(f"Pending Queue:   {remaining}")
    print("=" * 55)

    if delivery_stats["failed"] > 0 or remaining > 0:
        print("⚠️ Some messages failed delivery. Check cluster network and credentials.")
    else:
        print("🎉 All transactions delivered and confirmed by Confluent Cloud!")

## 6. Verify Fraud Alerts in `fraud-alerts`

Once the consumer processes the transactions, you can inspect the resulting fraud alerts in one of two ways:

### Option A: In Your Local Terminal
Your running `streaming.consumer` terminal will display:
```text
FRAUD ALERT: colab_high_val_... ['HIGH_VALUE_TRANSACTION'] Amount: ₹350,000.00
FRAUD ALERT: colab_burst_5_...  ['RAPID_REPEATED_TRANSFERS'] Amount: ₹5,000.00
```

### Option B: Right Here in Colab
Execute the cell below to read recent alerts directly from the `fraud-alerts` topic.

In [ ]:
# Step 6: Read Recent Alerts from fraud-alerts Topic
try:
    alert_consumer_conf = {
        "bootstrap.servers": BOOTSTRAP_SERVERS,
        "security.protocol": "SASL_SSL",
        "sasl.mechanisms": "PLAIN",
        "sasl.username": API_KEY,
        "sasl.password": API_SECRET,
        "group.id": f"colab-alert-reader-{int(time.time())}",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False,
    }
    alert_consumer = Consumer(alert_consumer_conf)

    # Query watermark offsets
    tp = TopicPartition(ALERT_TOPIC, 0)
    low, high = alert_consumer.get_watermark_offsets(tp, timeout=10)
    print(f"Topic '{ALERT_TOPIC}' partition 0: {high} total alerts recorded.\n")

    # Read latest alerts (up to 5)
    start_offset = max(0, high - 5)
    alert_consumer.assign([TopicPartition(ALERT_TOPIC, 0, start_offset)])

    print(f"Reading alerts from offset {start_offset} to {high}:")
    print("-" * 80)
    alerts_found = 0

    for _ in range(10):
        msg = alert_consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            continue
        alert = json.loads(msg.value().decode("utf-8"))
        alerts_found += 1
        print(f"🚨 Offset {msg.offset()}: TX={alert['transaction_id']}")
        print(f"   Reasons: {alert['risk_reasons']} | Amount: ₹{alert['amount']:,.2f}")
        print(f"   Sender:  {alert['sender_id']} ➔ Receiver: {alert['receiver_id']}")
        print("-" * 80)

    alert_consumer.close()
    if alerts_found == 0:
        print("No new alerts read. Ensure the consumer is running to process incoming transactions.")
except Exception as exc:
    print(f"Note: Alert inspection encountered: {exc}")

## 7. Troubleshooting & Cleanup

### Common Issues and Resolutions
1. **`1 Kafka message(s) still pending delivery`**:
   - Initial connection to Confluent Cloud brokers requires TLS handshake and `ApiVersionRequest`.
   - Ensure `delivery.timeout.ms` is set to `15000` (15s) and `flush(timeout=15.0)` is used.
2. **`SASL authentication error`**:
   - Verify that your API Key and Secret match your cluster credentials without leading/trailing whitespace.
3. **Consumer Not Triggering Rapid Transfers**:
   - Ensure all transactions from the burst sender use the **same `sender_id`** as the Kafka message key.
   - Verify that the timestamps of the 5 burst transfers fall within a 60-second window.
4. **Duplicate Alerts on Consumer Restart**:
   - RingShield implements **At-Least-Once Delivery**: alerts are published before input offsets are committed. If the consumer stops unexpectedly, uncommitted messages will be re-processed on restart.

---

### 🧹 Cleanup
When you are done with the demo:
- In your local consumer terminal, press `Ctrl+C` for graceful shutdown.
- In Google Colab, you can close or restart the runtime (`Runtime` -> `Disconnect and delete runtime`).